# Dutch IT Job Market: one dataset across the platform

Explore daily observed jobs in Data, Cloud, SAP, AI, Software and Security. Offline records are synthetic. Live observations depend on source permissions and coverage. No applicant or CV data is used.

## Local reproducible walkthrough

From a repository checkout, run `make jobs-demo`. Set `JOB_MARKET_OUTPUT` to that output directory. The SQLite section below reads the resulting database without any cloud credentials. On JupyterHub, upload that database or use the Trino section after the platform run completes.


In [ ]:
import os, sqlite3, json
from pathlib import Path

# Override this environment variable if the output lives elsewhere.
output = Path(os.environ.get("JOB_MARKET_OUTPUT", "examples/dutch-it-jobs/output")).expanduser()
db_path = output / "job_market.sqlite"
print("SQLite demo database:", db_path.resolve())


In [ ]:
if db_path.exists():
    with sqlite3.connect(db_path) as cx:
        rows = cx.execute("SELECT snapshot_date, jobs, companies FROM daily_market ORDER BY snapshot_date").fetchall()
    print(rows)
else:
    print("Run make jobs-demo from the repository root, then set JOB_MARKET_OUTPUT to its output directory.")


## Inspect original salary evidence and overlapping domain labels

The annualized range uses only EUR/year and EUR/month × 12. Hourly rates remain in original units. Estimated values must not be interpreted as employer-reported pay. A job can appear under multiple domains.


In [ ]:
if db_path.exists():
    with sqlite3.connect(db_path) as cx:
        cx.row_factory = sqlite3.Row
        latest = [dict(r) for r in cx.execute("SELECT * FROM latest_jobs ORDER BY title LIMIT 10")]
    for row in latest:
        job = json.loads(row["payload"])
        print(job["title"], job["city"], job["domains"],
              job["salary_min"], job["salary_max"], job["salary_currency"], job["salary_period"],
              "estimated" if job["salary_is_estimated"] else "reported/unknown")


## Run manifest and source coverage

A change in daily count may come from source failure or search limits. Snapshot absence does not establish closure. Inspect the manifest before drawing a market conclusion.


In [ ]:
manifests = sorted(output.glob("snapshot_date=*/manifest.json"))
if manifests:
    print(manifests[-1].read_text())


## The same use case through Trino

Run the following cells only on a configured ODAP Jupyter kernel with an authorized platform identity. Set `RUN_PLATFORM_QUERIES=True` to enable them. Business analyst perspectives read Gold; Bronze and Silver queries require their corresponding grants. Keep the purpose header/comment required by your deployment's access policy.


In [ ]:
RUN_PLATFORM_QUERIES = False
if RUN_PLATFORM_QUERIES:
    from odap_lab import trino
    display(trino.query("SELECT * FROM gold.job_market.mart_job_inventory LIMIT 20"))


In [ ]:
if RUN_PLATFORM_QUERIES:
    display(trino.query("SELECT * FROM gold.job_market.mart_job_demand_daily LIMIT 50"))
    display(trino.query("SELECT * FROM gold.job_market.mart_job_salary LIMIT 50"))
    display(trino.query("SELECT * FROM gold.job_market.mart_job_source_quality LIMIT 50"))


In [ ]:
# Engineers/stewards: enable only if authorized for these layers.
RUN_RESTRICTED_QUERIES = False
if RUN_PLATFORM_QUERIES and RUN_RESTRICTED_QUERIES:
    display(trino.query("SELECT snapshot_date, count(*) AS observations FROM bronze.job_market.observations GROUP BY 1 ORDER BY 1"))
    display(trino.query("SELECT * FROM silver.job_market.int_job_snapshots LIMIT 10"))


## Finish the platform walkthrough

1. Airflow: inspect `dutch_it_jobs_daily`, scheduled at 05:00 Europe/Amsterdam.
2. MinIO: inspect `odap-bronze/job-market/raw/snapshot_date=YYYY-MM-DD/` and the extraction manifest. Raw responses are restricted.
3. Superset: open dashboard `dutch-it-job-market` and compare skills, geography, work arrangements and salary coverage.
4. OpenMetadata: find `trino.gold.job_market.mart_job_inventory`; inspect definitions, owner, dbt tests and lineage to source observations.
5. Re-run one snapshot date and confirm that daily job identities remain unique.

All analysis is about observed advertisements. It is not a census, a wage survey, a candidate ranking or proof that a missing job has closed.
